# PHYT1D — File 1: Core Functions Library

All reusable functions for the PHYT1D digital twin pipeline.

**Contents:**
- VO2max prior estimation
- ODE simulator (`build_rates`, `simulate_trajectory_main`)
- MCMC Window A functions (`log_prior_A_main`, `log_likelihood_main`, `run_MCMC_A_main`)
- MCMC Window B functions (`log_prior_B_main`, `run_MCMC_B_main`)
- Uncertainty propagation (`draw_posterior_samples`)
- Evaluation metrics (`MARD_fn`, `RMSE_fn`, `TIR_fn`)

**Usage:** Run this file first. All other files depend on it via `%run phyt1d_functions.ipynb`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from scipy import stats
warnings.filterwarnings("ignore")
print("PHYT1D Functions Library — v2.0")
print("="*50)


## 1. VO2max Prior Estimation (Van de Poppe 2021)

In [ ]:
def estimate_VO2max(age_or_group, sex):
    """
    VO2max prior from LowLands Fitness Registry quadratic equations.
    Males  : VO2max = -0.0010*age² - 0.2248*age + 54.286  [mL/kg/min]
    Females: VO2max = -0.0021*age² - 0.1407*age + 43.066  [mL/kg/min]
    T1DS age groups: child=10yr, adolescent=15yr, adult=35yr
    """
    AGE_MAP = {"child": 10, "adolescent": 15, "adult": 35}
    age = AGE_MAP.get(str(age_or_group).lower(), age_or_group)           if isinstance(age_or_group, str) else float(age_or_group)
    if sex.upper() == "M":
        return max(-0.0010*age**2 - 0.2248*age + 54.286, 15.0)
    return max(-0.0021*age**2 - 0.1407*age + 43.066, 15.0)

print("✓ estimate_VO2max defined")


## 2. Fixed Parameters (Group D)

In [ ]:
FIXED = {
    "VI": 0.126, "ke": 0.127, "beta": 8.0,
    "f": 0.90, "VG": 1.45, "alpha": 7.0,
    "r1": 1.44, "r2": 0.81, "Gth": 60.0,
    "Fc01": 0.0097, "gamma_aer": 0.003, "gamma_res": 0.0015,
    "tau_ramp": 20.0, "delta_res": 0.003,
}
print("✓ FIXED parameters (Group D) loaded")


## 3. ODE Simulator

In [ ]:
def build_rates(user_params, dt=5.0, T=1440):
    """
    Build CHO ingestion and insulin (bolus + basal) rate arrays.

    Returns
    -------
    cho : np.ndarray — CHO rate [mg/kg/min] at each 5-min step
    ins : np.ndarray — total insulin rate [mU/kg/min] at each step
    """
    BW = 75.0
    n  = int(T / dt) + 1
    cho = np.zeros(n)
    ins = np.zeros(n)

    f  = user_params.get("CHO_factor",  1.0)
    bf = user_params.get("bolus_factor", 1.0)

    meals = [
        ("BF", user_params["t_BF"], user_params["CHO_BF"] * f),
        ("LU", user_params["t_LU"], user_params["CHO_LU"] * f),
        ("DN", user_params["t_DN"], user_params["CHO_DN"] * f),
        ("SN", user_params["t_SN"], user_params["CHO_SN"] * f),
    ]
    for _, tm, g in meals:
        idx = int(round(tm / dt))
        if 0 <= idx < n:
            cho[idx] += g * 1000 / BW / dt
            ins[idx] += (g / user_params["CR"]) * bf * 1000 / BW / dt

    # Continuous basal insulin — ~0.35 U/h default for T1D adult
    basal_U_per_h  = user_params.get("basal_rate", 0.35)
    basal_per_step = basal_U_per_h * 1000 / 60 * dt / BW
    ins += basal_per_step

    return cho, ins


def simulate_trajectory_main(theta_A, theta_B, user_params, fixed, dt=5.0, T=1440):
    """
    Integrate the augmented Dalla Man ODE (Euler, Δt=5 min) for one 24h day.

    Parameters
    ----------
    theta_A     : dict — Group A params {SI, SG, Gb, p2, kd, ka2, kempt, kabs}
    theta_B     : dict — Group B params {beta_aer, beta_res, tau_post, tau_on, VO2max, phi}
    user_params : dict — Group C params (exercise prescription, meals, bolus)
    fixed       : dict — Group D fixed params
    dt          : float — Euler step [min]
    T           : int   — simulation length [min]

    Returns
    -------
    times  : np.ndarray [min]
    IG_arr : np.ndarray [mg/dL] — interstitial glucose (CGM equivalent)
    G_arr  : np.ndarray [mg/dL] — plasma glucose
    """
    # ── Unpack Group A ────────────────────────────────────────────────────────
    SI    = theta_A["SI"];    SG    = theta_A["SG"];   Gb    = theta_A["Gb"]
    p2    = theta_A["p2"];    kd    = theta_A["kd"];   ka2   = theta_A["ka2"]
    kempt = theta_A["kempt"]; kabs  = theta_A["kabs"]

    # ── Unpack Group B ────────────────────────────────────────────────────────
    beta_aer = theta_B["beta_aer"]; beta_res = theta_B["beta_res"]
    tau_post = theta_B["tau_post"]; tau_on   = theta_B["tau_on"]
    phi_val  = theta_B.get("phi", 0.0)

    # ── Unpack Group D ────────────────────────────────────────────────────────
    VI        = fixed["VI"];       ke        = fixed["ke"]
    beta_d    = fixed["beta"];     f         = fixed["f"]
    VG        = fixed["VG"];       alpha     = fixed["alpha"]
    Fc01      = fixed["Fc01"];     gamma_aer = fixed["gamma_aer"]
    gamma_res = fixed["gamma_res"]; tau_ramp = fixed["tau_ramp"]
    delta_res_v = fixed["delta_res"]

    # ── Exercise prescription ─────────────────────────────────────────────────
    u       = user_params["u"]
    d       = user_params["d"]
    t_start = user_params["t_start"]
    ex_type = user_params["exercise_type"]
    t_end_ex = t_start + d

    # ── Build input arrays ────────────────────────────────────────────────────
    n_steps = int(T / dt) + 1
    times   = np.arange(n_steps) * dt
    cho_arr, ins_arr = build_rates(user_params, dt, T)

    # ── Correct basal steady-state initialisation (Dalla Man 2014) ───────────
    # All derivatives = 0 at steady state:
    #   ke*Ip = ka2*Isc2  →  Isc2b = Ipb*ke/ka2
    #   kd*Isc1 = ka2*Isc2  →  Isc1b = Ipb*ke/kd
    #   brate/VI = kd*Isc1  →  brate = Ipb*ke*VI
    Ipb   = 10.0                        # mU/L — physiological basal plasma insulin
    Isc2b = Ipb * ke / (ka2 + 1e-12)   # mU/kg
    Isc1b = Ipb * ke / (kd  + 1e-12)   # mU/kg
    brate = Ipb * ke * VI               # mU/kg/min — ~0.16 mU/kg/min ≈ 0.72 U/h
    BW    = 75.0

    state = dict(Isc1=Isc1b, Isc2=Isc2b, Ip=Ipb,
                 Qsto1=0., Qsto2=0., Qgut=0., G=Gb, X=0., IG=Gb)
    G_arr  = np.zeros(n_steps)
    IG_arr = np.zeros(n_steps)
    G_arr[0]  = Gb
    IG_arr[0] = Gb

    # ── Euler integration ─────────────────────────────────────────────────────
    for i in range(n_steps - 1):
        t = times[i]

        # ── Exercise state ────────────────────────────────────────────────────
        if u > 0 and t_start <= t <= t_end_ex:
            t_ex    = t - t_start
            t_prime = 0.0
            active  = True
        elif u > 0 and t > t_end_ex:
            t_ex    = d
            t_prime = t - t_end_ex
            active  = False
        else:
            t_ex = 0.0; t_prime = 0.0; active = False

        # ── Exercise ODE increments ───────────────────────────────────────────
        if u > 0 and active:
            if ex_type == "aerobic":
                # ΔSI_ex: exponential ramp-up (Riddell 2017)
                dSI_ex = beta_aer * u * (1.0 - np.exp(-t_ex / tau_on))
            else:
                # Resistance: transient SI decrease (Yardley 2013)
                dSI_ex = -delta_res_v * u * min(t_ex / tau_ramp, 1.0)
            ramp    = min(t_ex / tau_ramp, 1.0)
            gcoef   = gamma_aer if ex_type == "aerobic" else gamma_res
            # ΔFc01: non-insulin-mediated muscle uptake (Dalla Man 2009)
            Fc01_e  = Fc01 + gcoef * u * ramp
            # kempt_eff: gastric emptying acceleration (Riddell 2017)
            ke_emp  = kempt * (1.0 + phi_val * u)
            # ΔEGP: resistance EGP surge only (Yardley 2013)
            dEGP    = beta_res * u**2 * ramp if ex_type == "resistance" else 0.0
        else:
            dSI_ex = 0.; Fc01_e = Fc01; ke_emp = kempt; dEGP = 0.

        # ── Post-exercise SI tail (Hinshaw 2013) ──────────────────────────────
        dSI_tail = beta_aer * u * np.exp(-t_prime / tau_post)                    if (u > 0 and t_prime > 0) else 0.0
        SI_e = SI * (1.0 + dSI_ex + dSI_tail)

        # ── Insulin input (SC absorption delay = beta_d min) ─────────────────
        delay_idx = max(0, i - int(beta_d / dt))
        I_in = ins_arr[delay_idx] + brate

        # ── Unpack state ──────────────────────────────────────────────────────
        Isc1  = state["Isc1"]; Isc2 = state["Isc2"]; Ip = state["Ip"]
        Qsto1 = state["Qsto1"]; Qsto2 = state["Qsto2"]; Qgut = state["Qgut"]
        G = state["G"]; X = state["X"]; IG = state["IG"]

        # ── ODE updates ───────────────────────────────────────────────────────
        # SC insulin PK (Schiavon 2018)
        state["Isc1"] = max(Isc1 + dt * (-kd * Isc1 + I_in / VI), 0.)
        state["Isc2"] = max(Isc2 + dt * (kd * Isc1 - ka2 * Isc2), 0.)
        state["Ip"]   = max(Ip   + dt * (ka2 * Isc2 - ke * Ip),   0.)

        # CHO absorption (Dalla Man 2006)
        state["Qsto1"] = max(Qsto1 + dt * (-ke_emp * Qsto1 + cho_arr[i]), 0.)
        state["Qsto2"] = max(Qsto2 + dt * (ke_emp * Qsto1 - ke_emp * Qsto2), 0.)
        state["Qgut"]  = max(Qgut  + dt * (ke_emp * Qsto2 - kabs * Qgut),   0.)
        Ra_val = f * kabs * Qgut

        # Glucose-insulin kinetics (Dalla Man 2014)
        # NOTE: rho(G) is a risk-scoring function ONLY — not used inside ODE
        # X acts unconditionally on glucose disposal
        Fc01_mg = Fc01_e * 18.0 / VG          # mmol/kg/min → mg/dL/min
        state["G"]  = max(G  + dt * (-(SG + X) * G + SG * Gb
                                     + Ra_val / VG + dEGP - Fc01_mg), 20.)
        state["X"]  = X  + dt * (-p2 * (X - SI_e * (Ip - Ipb)))
        state["IG"] = max(IG + dt * (-(1.0 / alpha) * (IG - G)), 20.)

        G_arr[i+1]  = state["G"]
        IG_arr[i+1] = state["IG"]

    return times, IG_arr, G_arr

print("✓ build_rates defined")
print("✓ simulate_trajectory_main defined")


## 4. MCMC Window A Functions

In [ ]:
def log_prior_A_main(theta):
    """Log prior for Group A parameters."""
    priors = {
        "SI"   : ("lognormal", -5.3,  0.6),
        "SG"   : ("normal",    0.025, 0.013),
        "Gb"   : ("normal",    120.,  10.),
        "p2"   : ("normal",    0.025, 0.015),   # lowered from 0.05
        "kd"   : ("normal",    0.026, 0.013),
        "ka2"  : ("normal",    0.066, 0.030),
        "kempt": ("normal",    0.055, 0.020),
        "kabs" : ("normal",    0.057, 0.020),
    }
    lp = 0.
    for name, val in theta.items():
        if val <= 0: return -np.inf
        k = priors[name][0]
        if k == "lognormal":
            lp += stats.norm.logpdf(np.log(val), priors[name][1],
                                    priors[name][2]) - np.log(val)
        else:
            lp += stats.norm.logpdf(val, priors[name][1], priors[name][2])
    return lp


def log_likelihood_main(theta_A, theta_B, cgm_obs, times_obs,
                         user_params, fixed, sigma=5.0):
    """Gaussian log-likelihood on CGM residuals."""
    try:
        times_sim, ig_sim, _ = simulate_trajectory_main(
            theta_A, theta_B, user_params, fixed)
        ig_at_obs = np.interp(times_obs, times_sim, ig_sim)
        residuals  = cgm_obs - ig_at_obs
        return (-0.5 * np.sum((residuals / sigma)**2)
                - len(cgm_obs) * np.log(sigma * np.sqrt(2 * np.pi)))
    except Exception:
        return -np.inf


def run_MCMC_A_main(theta_init, theta_B_fixed, cgm_obs, times_obs,
                    user_params, fixed,
                    n_samples=2000, burn_in=1000, rng=None, verbose=True):
    """
    Adaptive SCMH for Group A parameters.
    SI proposed in log-space (lognormal prior — prevents chain collapse).
    """
    if rng is None:
        rng = np.random.default_rng(0)

    LOG_SPACE = {"SI"}   # parameters proposed in log-space

    pnames    = list(theta_init.keys())
    theta_cur = dict(theta_init)
    lp_cur    = log_prior_A_main(theta_cur) + log_likelihood_main(
        theta_cur, theta_B_fixed, cgm_obs, times_obs, user_params, fixed)

    # Initial proposal σ — tighter for log-space params
    sigma_prop = {k: abs(v) * 0.05 + 1e-8 for k, v in theta_cur.items()}
    sigma_prop["SI"] = 0.15   # log-space sigma

    accept_cnt = {k: 0 for k in pnames}
    total_cnt  = {k: 0 for k in pnames}
    chain = []

    for i in range(n_samples):
        for name in pnames:
            theta_prop = dict(theta_cur)

            if name in LOG_SPACE:
                # Propose in log-space — symmetric on log scale
                log_cur          = np.log(theta_cur[name])
                log_prop         = log_cur + rng.normal(0, sigma_prop[name])
                theta_prop[name] = np.exp(log_prop)
            else:
                theta_prop[name] = theta_cur[name] + rng.normal(0, sigma_prop[name])

            if theta_prop[name] <= 0:
                total_cnt[name] += 1
                continue

            lp_prop = log_prior_A_main(theta_prop) + log_likelihood_main(
                theta_prop, theta_B_fixed, cgm_obs, times_obs, user_params, fixed)

            if np.log(rng.uniform()) < lp_prop - lp_cur:
                theta_cur  = theta_prop
                lp_cur     = lp_prop
                accept_cnt[name] += 1
            total_cnt[name] += 1

        # Adapt proposal σ during burn-in
        if i < burn_in and (i + 1) % 50 == 0:
            for name in pnames:
                r = accept_cnt[name] / max(total_cnt[name], 1)
                if r > 0.28:
                    sigma_prop[name] *= 1.1
                elif r < 0.18:
                    sigma_prop[name] *= 0.9

        if i >= burn_in:
            chain.append(dict(theta_cur))

        if verbose and (i + 1) % 200 == 0:
            avg = np.mean([accept_cnt[k] / max(total_cnt[k], 1)
                           for k in pnames])
            print(f"  Window A  {i+1:>5d}/{n_samples}  accept={avg:.2f}  lp={lp_cur:.1f}")

    print()
    return chain

print("✓ log_prior_A_main defined")
print("✓ log_likelihood_main defined")
print("✓ run_MCMC_A_main defined")


## 5. MCMC Window B Functions

In [ ]:
def log_prior_B_main(theta_B, vo2_mean=45.2):
    """Log prior for Group B exercise parameters."""
    priors = {
        "beta_aer": ("normal",  0.50,     0.15),
        "beta_res": ("normal",  0.30,     0.10),
        "tau_post": ("gamma",   4.0,      60.0),   # mean ~ 240 min
        "tau_on"  : ("normal",  15.0,     5.0),
        "VO2max"  : ("normal",  vo2_mean, 8.0),
        "phi"     : ("uniform", 0.0,      0.5),
    }
    lp = 0.
    for name, val in theta_B.items():
        k = priors[name][0]
        if k == "normal":
            lp += stats.norm.logpdf(val, priors[name][1], priors[name][2])
        elif k == "gamma":
            if val <= 0: return -np.inf
            lp += stats.gamma.logpdf(val, a=priors[name][1], scale=priors[name][2])
        elif k == "uniform":
            if not (priors[name][1] <= val <= priors[name][2]):
                return -np.inf
    for n in ["beta_aer", "beta_res", "tau_post", "tau_on", "VO2max"]:
        if theta_B.get(n, 1.) <= 0:
            return -np.inf
    return lp


def run_MCMC_B_main(theta_B_init, theta_A_med, cgm_obs_ex, times_ex,
                    user_params, fixed, vo2_mean=45.2,
                    n_samples=2000, burn_in=1000, rng=None, verbose=True):
    """Adaptive SCMH for Group B exercise parameters."""
    if rng is None:
        rng = np.random.default_rng(1)

    pnames    = list(theta_B_init.keys())
    theta_cur = dict(theta_B_init)
    lp_cur    = (log_prior_B_main(theta_cur, vo2_mean) +
                 log_likelihood_main(theta_A_med, theta_cur, cgm_obs_ex,
                                     times_ex, user_params, fixed))

    sigma_prop = {k: abs(v) * 0.10 + 1e-8 for k, v in theta_cur.items()}
    accept_cnt = {k: 0 for k in pnames}
    total_cnt  = {k: 0 for k in pnames}
    chain = []

    for i in range(n_samples):
        for name in pnames:
            theta_prop = dict(theta_cur)
            theta_prop[name] = theta_cur[name] + rng.normal(0, sigma_prop[name])
            lp_prop = (log_prior_B_main(theta_prop, vo2_mean) +
                       log_likelihood_main(theta_A_med, theta_prop, cgm_obs_ex,
                                           times_ex, user_params, fixed))
            if np.log(rng.uniform()) < lp_prop - lp_cur:
                theta_cur  = theta_prop
                lp_cur     = lp_prop
                accept_cnt[name] += 1
            total_cnt[name] += 1

        if i < burn_in and (i + 1) % 50 == 0:
            for name in pnames:
                r = accept_cnt[name] / max(total_cnt[name], 1)
                if r > 0.28:
                    sigma_prop[name] *= 1.1
                elif r < 0.18:
                    sigma_prop[name] *= 0.9

        if i >= burn_in:
            chain.append(dict(theta_cur))

        if verbose and (i + 1) % 200 == 0:
            avg = np.mean([accept_cnt[k] / max(total_cnt[k], 1)
                           for k in pnames])
            print(f"  Window B  {i+1:>5d}/{n_samples}  accept={avg:.2f}")

    print()
    return chain

print("✓ log_prior_B_main defined")
print("✓ run_MCMC_B_main defined")


## 6. Uncertainty Propagation

In [ ]:
def draw_posterior_samples(chain, n, rng=None):
    """Sample n parameter dicts from MCMC chain with replacement."""
    if rng is None:
        rng = np.random.default_rng(99)
    idx = rng.integers(0, len(chain), n)
    return [chain[i] for i in idx]


def run_ensemble(chain_A, chain_B, user_params, fixed,
                 n_real=200, dt=5.0, T=1440, rng=None):
    """
    Run n_real posterior realisations and return median + 25/75th CI.

    Returns
    -------
    times  : np.ndarray [min]
    IG_50  : median IG [mg/dL]
    IG_25  : 25th percentile
    IG_75  : 75th percentile
    all_IG : (n_valid, n_steps) array of all trajectories
    """
    if rng is None:
        rng = np.random.default_rng(99)

    samples_A = draw_posterior_samples(chain_A, n_real, rng)
    samples_B = draw_posterior_samples(chain_B, n_real, rng)

    n_steps = int(T / dt) + 1
    all_IG  = np.zeros((n_real, n_steps))
    times   = np.arange(n_steps) * dt

    print(f"Running {n_real} posterior realisations...")
    for r in range(n_real):
        tA = samples_A[r]; tB = samples_B[r]
        try:
            _, ig, _ = simulate_trajectory_main(tA, tB, user_params, fixed, dt, T)
            all_IG[r] = ig
        except Exception:
            all_IG[r] = np.nan
        if (r + 1) % 50 == 0:
            print(f"  {r+1}/{n_real}")

    valid  = ~np.isnan(all_IG).any(axis=1)
    all_IG = all_IG[valid]
    print(f"\u2713 {valid.sum()}/{n_real} realisations valid.")

    IG_50 = np.percentile(all_IG, 50, axis=0)
    IG_25 = np.percentile(all_IG, 25, axis=0)
    IG_75 = np.percentile(all_IG, 75, axis=0)

    return times, IG_50, IG_25, IG_75, all_IG

print("✓ draw_posterior_samples defined")
print("✓ run_ensemble defined")


## 7. Evaluation Metrics

In [ ]:
def MARD_fn(ig_true, ig_sim):
    """
    Mean Absolute Relative Difference [%].
    Pass threshold: MARD < 10% (Cappon 2023).
    """
    return float(np.mean(
        np.abs(np.array(ig_true) - np.array(ig_sim)) / np.array(ig_true)
    ) * 100.0)


def RMSE_fn(ig_true, ig_sim):
    """Root Mean Square Error [mg/dL]."""
    return float(np.sqrt(np.mean(
        (np.array(ig_true) - np.array(ig_sim))**2
    )))


def TIR_fn(ig):
    """
    Time-in-Range metrics (Battelino 2019).
    TIR: 70–180 mg/dL | TBR: < 70 mg/dL | TAR: > 180 mg/dL
    """
    n   = len(ig)
    tir = np.sum((ig >= 70) & (ig <= 180)) / n * 100.0
    tbr = np.sum(ig < 70)                  / n * 100.0
    tar = np.sum(ig > 180)                 / n * 100.0
    return {"TIR": round(tir, 1), "TBR": round(tbr, 1), "TAR": round(tar, 1)}


def print_eval_report(ig_true, IG_50, THETA_B_MED, THETA_B_TRUE,
                      chain_A, chain_B):
    """Print full evaluation report."""
    mard = MARD_fn(ig_true, IG_50)
    rmse = RMSE_fn(ig_true, IG_50)
    tir  = TIR_fn(IG_50)

    print("\n" + "="*55)
    print("  PHYT1D — Evaluation Results")
    print("="*55)
    print(f"  MARD : {mard:.2f}%  "
          f"{'PASS ✓' if mard < 10 else 'FAIL ✗'}  (threshold < 10%)")
    print(f"  RMSE : {rmse:.2f} mg/dL")
    print(f"  TIR  : {tir['TIR']:.1f}%  "
          f"TBR: {tir['TBR']:.1f}%  TAR: {tir['TAR']:.1f}%")

    vo2_hat = THETA_B_MED["VO2max"]
    vo2_re  = abs(vo2_hat - THETA_B_TRUE["VO2max"]) / THETA_B_TRUE["VO2max"] * 100
    print(f"  VO2max recovery: {vo2_re:.1f}%  "
          f"{'PASS ✓' if vo2_re < 15 else 'FAIL ✗'}")

    tp_hat = THETA_B_MED["tau_post"]
    tp_re  = abs(tp_hat - THETA_B_TRUE["tau_post"]) / THETA_B_TRUE["tau_post"] * 100
    print(f"  tau_post: est={tp_hat:.1f}  "
          f"true={THETA_B_TRUE['tau_post']:.1f}  RE={tp_re:.1f}%")
    print("="*55)

    return {"MARD": mard, "RMSE": rmse, "TIR": tir,
            "vo2_re": vo2_re, "tp_re": tp_re}

print("✓ MARD_fn, RMSE_fn, TIR_fn, print_eval_report defined")
print()
print("="*50)
print("✓ ALL PHYT1D FUNCTIONS LOADED SUCCESSFULLY")
print("  Import into other notebooks with:  %run phyt1d_functions.ipynb")
print("="*50)
